**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**03 | EEG and Gaze Preprocessing, Features and Neurophysiology**

![Observed EEG and gaze feature engineering](../../images/03_preprocessing_features_and_neurophysiology.svg)

**Research question:** Can matched EEG and gaze features be extracted from the same eligible ZuCo 2.0 windows without losing source identities or participant-level separation?

**Objective:** Extract EEG spectral/RMS and gaze/event summaries in six two-second bins per 12-second window; save aligned per-window means and temporal arrays; inspect one Train-only fixation-locked raw EEG example.

**Dataset:** ZuCo 2.0, NEMAR `nm000279` v1.0.0; **six people and 12 original NR/TSR recordings**, with fixed **3 Train / 1 Validation / 2 reserved Test** participant roles.

**Recorded results:** All **206** Stage-02 windows produced both modalities' features, **zero** window exclusions and **zero** non-finite window-mean feature values. Each temporal representation has shape **(206, 6, 4)**. The resulting partition counts remain **95 Train, 31 Validation and 80 reserved Test windows**.

**Scientific boundary:** These are descriptive features, **not** task-classification results or evidence of cognitive processing. EEG/eye alignment remains based on BIDS timing metadata, independent clock drift has not been verified and the current six-person cohort is insufficient for reliable generalisation conclusions.

**Expanded-cohort implementation (pending execution):** Feature extraction remains source-window aligned and uses the participant roles saved by Notebook 01. A new Train-derived per-feature quality table records missingness and constant predictors without changing the approved signal processing or model inputs. The existing displayed outputs remain historical six-person pilot observations until this stage is rerun.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------- 1. Isolated Development Checkout ----------------------
from pathlib import Path
import subprocess
import sys

repo_url = (
    "https://github.com/RezaSaadatyar/"
    "Eye-Tracking-Neurocognitive-Intelligence.git"
)
repo_path = Path("/content/Eye-Tracking-Neurocognitive-Intelligence-fusion-expanded")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", "--branch", "main", "--single-branch",
         repo_url, str(repo_path)], check=True
    )
elif not (repo_path / ".git").is_dir():
    raise RuntimeError("Development checkout path is not a Git repository")

# Refuse to combine a newer remote stage with unrefreshed local code.
# Local output files or user edits are never deleted or reset.
subprocess.run(
    ["git", "fetch", "-q", "--no-tags", "origin", "main"],
    cwd=repo_path, check=True
)
local_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo_path, text=True).strip()
remote_sha = subprocess.check_output(
    ["git", "rev-parse", "FETCH_HEAD"], cwd=repo_path, text=True).strip()
if local_sha != remote_sha:
    raise RuntimeError(
        "This Colab checkout is behind or diverges from main. "
        "Preserve/publish any local outputs, then open the notebook "
        "in a fresh Runtime. No files have been reset or deleted."
    )

# Prevent cached research modules from another checkout being reused.
for module_name in (
    "zuco_io", "alignment", "features", "models",
    "core.plot_style", "core.output_download",
    "python_modules.core.colab_setup",
    "python_modules.core.repository_sync",
):
    loaded = sys.modules.get(module_name)
    module_file = getattr(loaded, "__file__", None)
    if module_file and not Path(module_file).resolve().is_relative_to(
            repo_path.resolve()):
        raise RuntimeError(
            "Python has research modules from another checkout. "
            "Start in a fresh Runtime to avoid stale imports."
        )
print("Fusion development checkout:", local_sha[:12])

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


RuntimeError: This Colab checkout is behind or diverges from main. Preserve/publish any local outputs, then open the notebook in a fresh Runtime. No files have been reset or deleted.

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion", sync_mode="off", module_refresh="off")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import sys

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 6. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from zuco_io import open_run, session
from alignment import ensure_disjoint_registry
from features import (
    EEG_FIELDS, GAZE_FIELDS, eeg_channels, extract_window, clean_eeg,
)

# ---------------------------- 7. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 8. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Verify the Frozen Paired-Window Protocol**

The recorded Stage-02 registry supplies **206 unique eligible 12-second windows**: **101 NR / 105 TSR**. Every accepted source ID and its Train/Validation/Test role is matched to the Notebook 01 manifest before feature extraction.

**Observed:** The input is participant-disjoint and has 95 Train, 31 Validation and 80 reserved Test windows. The original 12 Run-level timing records all mark independently verified clock drift as **False**. These source-quality limitations carry into later model comparisons.


In [ ]:
# ============= 02. Verify Paired Windows and Modality Contracts ==============

required = [
    "01_source_manifest.csv", "01_protocol.json",
    "02_window_registry.csv", "02_alignment_quality.csv",
]
missing = [p for p in required if not (output_dir / p).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing upstream source or alignment outputs: {missing}"
    )

source = pd.read_csv(
    output_dir / "01_source_manifest.csv",
    dtype={"participant_id": str},
)
registry = pd.read_csv(
    output_dir / "02_window_registry.csv",
    dtype={"participant_id": str},
)
alignment_quality = pd.read_csv(
    output_dir / "02_alignment_quality.csv",
    dtype={"participant_id": str},
)
protocol = json.loads((output_dir / "01_protocol.json").read_text())

if (protocol.get('selected_people') != source.participant_id.nunique()
        or set(registry.participant_id) != set(source.participant_id)):
    raise ValueError(
        'Stage-01/02 cohorts differ. Rerun the changed upstream stage '
        'before extracting features.')
ensure_disjoint_registry(registry)
selected = registry.loc[registry.eligible].copy()
if selected.empty:
    raise ValueError("No eligible paired windows from Notebook 02")

# A selected window must belong to its original source run and split.
join_keys = ["participant_id", "task", "run"]
source_roles = source[join_keys + ["split"]].drop_duplicates()
linked = selected.merge(
    source_roles, on=join_keys, how="left",
    suffixes=("_window", "_source"), validate="many_to_one",
)
if (linked.split_source.isna().any() or not
        linked.split_window.eq(linked.split_source).all()):
    raise ValueError("Window source identities or participant roles differ")

window_length = float(protocol["window_seconds"])
bin_seconds = float(protocol["bin_seconds"])
bins_ratio = window_length / bin_seconds
if not np.isclose(bins_ratio, round(bins_ratio)):
    raise ValueError("Window length must divide into equal feature bins")
BINS = int(round(bins_ratio))
REQUEST = session()

print("Recorded source participants:", source.participant_id.nunique())
print("Recorded original runs:", len(source))
print("Eligible paired windows:", len(selected))
print("Feature bins per window:", BINS)
print("Independent clock drift verified:",
      alignment_quality.drift_verified.astype(str).str.lower().eq(
          "true").all())
display(
    selected.groupby(["split", "task"])
    .agg(windows=("window_id", "size"),
         participants=("participant_id", "nunique"))
    .reset_index()
)


**03. Extract EEG and Gaze Features from Matched Windows**

Use the fixed subset of **16 native EEG channels** where available. Apply within-window mean reference and a **1–40 Hz** bandpass. For each of six 2-second bins, calculate theta/alpha/beta log bandpower and EEG RMS. Separately calculate finite gaze-coordinate fraction, median gaze speed, pupil median and fixation-event occupancy from EyeLink records.

**Observed:** **206 of 206 windows** produced matching EEG and gaze sequences; **no extraction exceptions** were recorded. The saved features retain original participant/task/run/split/window identifiers and the selected feature units. No population-level or cortical localisation claims follow from this compact electrode subset.


In [ ]:
# ============== 03. Extract Matched Modality-Separated Features ==============

feature_rows, signal_windows, gaze_windows, accepted, issues = (
    [], [], [], [], [])
for _, run in source.iterrows():
    subset = selected.loc[
        selected.participant_id.eq(run.participant_id) &
        selected.task.eq(run.task) &
        selected.run.eq(run.run)]
    if subset.empty:
        continue
    preceding_ok = len(accepted)
    preceding_issues = len(issues)
    raw, eye, events, eye_meta, _ = open_run(run, raw_dir, REQUEST)
    channels = eeg_channels(raw, count=16)
    try:
        for _, row in subset.iterrows():
            try:
                eeg_seq, gaze_seq = extract_window(
                    raw, eye, events, eye_meta, row, channels,
                    bin_seconds=float(protocol['bin_seconds']))
                if eeg_seq.shape != (BINS, len(EEG_FIELDS)):
                    raise ValueError('EEG feature shape mismatch')
                if gaze_seq.shape != (BINS, len(GAZE_FIELDS)):
                    raise ValueError('Gaze feature shape mismatch')
                fields = {name: float(np.nanmean(eeg_seq[:, j]))
                          for j, name in enumerate(EEG_FIELDS)}
                fields.update({
                    name: float(np.nanmean(gaze_seq[:, j]))
                    for j, name in enumerate(GAZE_FIELDS)})
                feature_rows.append({**row.to_dict(), **{
                    'eeg_' + name: value
                    for name, value in fields.items()
                    if name in EEG_FIELDS}, **{
                    'gaze_' + name: value
                    for name, value in fields.items()
                    if name in GAZE_FIELDS}})
                signal_windows.append(eeg_seq)
                gaze_windows.append(gaze_seq)
                accepted.append(str(row.window_id))
            except (ValueError, FloatingPointError) as exc:
                issues.append({'window_id': row.window_id,
                               'reason': str(exc)})
    finally:
        raw.close()
    print(
        "Processed:", run.participant_id, run.task,
        "| candidate windows:", len(subset),
        "| retained:", len(accepted) - preceding_ok,
        "| excluded:", len(issues) - preceding_issues,
    )
features = pd.DataFrame(feature_rows)
if features.empty:
    raise RuntimeError('No paired temporal features were produced')
if features.window_id.duplicated().any():
    raise ValueError("Feature extraction duplicated original window IDs")
print("Extracted paired feature windows:", len(features))
print("Feature extraction issues:", len(issues))
display(features.head())


**04. Verify Feature Completeness and Participant Support**

The saved `03_modality_quality.csv` preserves all **12 original participant/task groups**, including the short `sub-YAK / NR` run with **one** window. No Run or participant was silently replaced.

**Observed:** `03_exclusion_reasons.csv` contains only its headers, indicating **zero feature-extraction exclusions**. All eight window-mean EEG/gaze feature columns are finite in all **206** rows.

**Interpretation:** `gaze_valid_fraction` equals **1.000 in every row** and therefore does not distinguish NR from TSR within this selected cohort. That constancy is only evidence of finite x/y samples, not of independently verified eye-tracker calibration, pupil quality or artefact freedom.


In [ ]:
# ============ 04. Check EEG/Gaze Completeness and Split Coverage =============

quality = pd.DataFrame(issues, columns=['window_id', 'reason'])
group_support = (features.groupby(['participant_id', 'split', 'task'])
                 .size().rename('n_windows').reset_index())
eligible_people = features.groupby('participant_id').task.nunique()
if not eligible_people.eq(2).all():
    raise ValueError('A participant lost one task after processing')
if set(features.split) != {'train', 'validation', 'test'}:
    raise ValueError('A split role is absent from processed windows')
display(group_support)
print("Excluded after feature extraction:", len(quality))
if not quality.empty:
    display(quality.groupby("reason").size().rename("n_windows"))
eeg_fields = ["eeg_" + name for name in EEG_FIELDS]
gaze_fields = ["gaze_" + name for name in GAZE_FIELDS]
print("Missing/non-finite EEG summary values:",
      int((~np.isfinite(features[eeg_fields])).sum().sum()))
print("Missing/non-finite gaze summary values:",
      int((~np.isfinite(features[gaze_fields])).sum().sum()))

# Assess Train-fittable features without using held-out distributions to
# define a transformation. Constant features are noted, not silently dropped.
feature_checks = []
train_rows = features.loc[features.split.eq('train')]
for field in eeg_fields + gaze_fields:
    train_values = train_rows[field].to_numpy(dtype=float)
    all_values = features[field].to_numpy(dtype=float)
    finite_train = train_values[np.isfinite(train_values)]
    if not len(finite_train):
        raise ValueError(
            f'No finite Train samples for feature {field}; review extraction')
    feature_checks.append({
        'feature': field,
        'modality': 'eeg' if field.startswith('eeg_') else 'gaze',
        'train_windows': len(train_values),
        'train_non_finite': int((~np.isfinite(train_values)).sum()),
        'all_windows': len(all_values),
        'all_non_finite': int((~np.isfinite(all_values)).sum()),
        'train_std': float(np.std(finite_train)),
        'constant_on_train': bool(np.std(finite_train) <= 1e-12),
    })
feature_quality = pd.DataFrame(feature_checks)
display(feature_quality)
print('Features constant on Train:',
      feature_quality.loc[feature_quality.constant_on_train,
                          'feature'].tolist())


**05. Train-Only Fixation-Locked EEG Diagnostic**

The recorded diagnostic uses fixation events and a **single raw EEG channel** from a Train participant. Its displayed raw voltage has a large baseline offset (approximately **−10,100 μV**), because the illustrative trace is not the referenced/filtered EEG feature signal and is not baseline-corrected.

**Scientific interpretation:** The trace is a source-reading diagnostic only; its offset and oscillations must **not** be interpreted as a valid fixation-related potential, event-related component, cortical source or cognitive response. No independent synchrony, artefact rejection or event-overlap deconvolution was established. The figure is not a model predictor.


In [ ]:
# ======== 05. Fixation-Locked EEG: Descriptive Train-Only Diagnostic =========

training_run = source.loc[source.split.eq('train')].iloc[0]
raw, eye, events, eye_meta, _ = open_run(
    training_run, raw_dir, REQUEST)
try:
    chans = eeg_channels(raw, count=16)
    starts = pd.to_numeric(
        events.loc[events.trial_type.eq('fixation'), 'onset'],
        errors='coerce').dropna().to_numpy()
    starts = starts[(starts > 0.2) &
                    (starts < raw.times[-1] - 0.6)][:80]
    sf = float(raw.info['sfreq'])
    samples = []
    for onset in starts:
        seg = raw.get_data(picks=chans[:1],
                           start=int((onset-0.2)*sf),
                           stop=int((onset+0.6)*sf))[0]
        if len(seg) == int(0.8 * sf) and np.isfinite(seg).all():
            samples.append(seg)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if samples:
        epochs = np.stack(samples)
        avg = np.mean(epochs, axis=0)
        t = np.arange(len(avg))/sf - 0.2
        ax.plot(t, avg*1e6)
        ax.axvline(0, color='black', ls='--', lw=1)
    ax.set(title='Exploratory fixation-locked EEG (Train only)',
           xlabel='Seconds from fixation onset', ylabel='EEG (µV)')
    fig.tight_layout()
    fig.savefig(output_dir / '03_fixation_locked_diagnostic.png',
                dpi=170)
    plt.show()
finally:
    raw.close()


**06. Save Paired Window Features and Temporal Evidence**

Six actual Stage-03 outputs are saved under `output/`:

- `03_features.csv`: **206 paired windows**, eight EEG/gaze mean features and original source/role fields.
- `03_temporal_sequences.npz`: EEG and gaze arrays with recorded shapes **(206, 6, 4)** each and matching source identifiers.
- `03_exclusion_reasons.csv`: header only; **zero** excluded windows.
- `03_modality_quality.csv`: retained counts for **12 participant/task groups**.
- `03_fixation_locked_diagnostic.png`: exploratory unfiltered single-channel raw EEG trace, **not** a validated FRP.
- `03_temporal_example.png`: a shared-window EEG/gaze feature illustration.

The NPZ producer writes source identifiers as NumPy Unicode arrays (rather than object dtype) for safe loading with `allow_pickle=False` on subsequent executions. The temporal figure displays separately standardised **within-window** feature trends to avoid combining incompatible physical units. **The saved predictors retain their original numeric values**, and any learned imputation or scaling belongs to later Train-fitted model pipelines.


In [ ]:
# ====== 06. Save Matched Features and Compact Temporal Representations =======

eeg_seq = np.stack(signal_windows).astype(np.float32)
gaze_seq = np.stack(gaze_windows).astype(np.float32)
if (len(features) != len(accepted) or
        not np.array_equal(features.window_id.astype(str), accepted)):
    raise ValueError('Feature/sequence identity mismatch')
features.to_csv(output_dir / '03_features.csv', index=False)
quality.to_csv(output_dir / '03_exclusion_reasons.csv', index=False)
group_support.to_csv(output_dir / '03_modality_quality.csv',
                     index=False)
feature_quality.to_csv(
    output_dir / '03_feature_quality.csv', index=False)
np.savez_compressed(
    output_dir / '03_temporal_sequences.npz', eeg=eeg_seq,
    gaze=gaze_seq,
    window_id=features.window_id.to_numpy(dtype=str),
    participant_id=features.participant_id.to_numpy(dtype=str),
    task=features.task.to_numpy(dtype=str),
    split=features.split.to_numpy(dtype=str))
# This display standardises each feature within one sample window.
# Saved arrays retain their original, unscaled units for later modelling.
def _display_relative_feature_trends(ax, seq, names, title):
    steps = np.arange(1, seq.shape[0] + 1)
    for col, name in enumerate(names):
        values = np.asarray(seq[:, col], dtype=float)
        available = np.isfinite(values)
        if not available.any():
            continue
        centre = np.mean(values[available])
        spread = np.std(values[available])
        relative = np.full(len(values), np.nan, dtype=float)
        relative[available] = (
            (values[available] - centre) / spread
            if spread > 0 else 0.
        )
        ax.plot(steps, relative, marker="o", label=name)
    ax.set(
        title=title,
        xlabel="2-second feature bin",
        ylabel="Within-window z-score (display only)",
        xticks=steps,
    )
    if ax.lines:
        ax.legend(fontsize=7, loc="best")


fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
_display_relative_feature_trends(
    axes[0], eeg_seq[0], EEG_FIELDS, "EEG features: one paired window"
)
_display_relative_feature_trends(
    axes[1], gaze_seq[0], GAZE_FIELDS, "Gaze features: same window"
)
fig.tight_layout()
fig.savefig(output_dir / "03_temporal_example.png", dpi=170)
plt.show()
print("Saved original-unit arrays:", eeg_seq.shape, gaze_seq.shape)


**07. Interpretation and Next Dependency**

The entire **206-window** Stage-02 registry survived feature extraction: **95 Train (41 NR, 54 TSR)**, **31 Validation (20 NR, 11 TSR)** and **80 reserved Test (40 NR, 40 TSR)**. There are **three independent Train people**, **one Validation person** and **two Test people**.

The eight feature summaries have no non-finite entries; `gaze_valid_fraction` has no variation in this cohort. These results support a **technical comparison of NR versus TSR**, not an estimate of reading comprehension or model generalisation.

**Next:** Notebook 04 trains classical EEG-only, gaze-only and simple-fusion comparators exclusively on Train and reports **Validation** performance. The Test partition remains untouched for model choice.


In [ ]:
# ================== 07. Interpretation and Next Dependency ===================

print("Next: 04_unimodal_baselines_and_classical_fusion.ipynb")
download_outputs_from_colab("eeg_eye_tracking_fusion/output", "03")
